# Splitting data in train, validation, test

### Import libraries and tidied data

In [1]:
from scipy.sparse import load_npz
import numpy as np

sparse_matrix = load_npz('data/tidied_360k/user_artist_matrix.npz')
users = np.load('data/tidied_360k/users.npy', allow_pickle=True)
artists = np.load('data/tidied_360k/artists.npy', allow_pickle=True)

In [2]:
print(artists[3])

0003ccd0-27cb-4e5d-9fea-a44c8111c1d9


In [10]:
print(users[0])

Babs_05


In [13]:
print(sparse_matrix)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 51790 stored elements and shape (11, 22823)>
  Coords	Values
  (0, 3)	1
  (0, 6)	7
  (0, 7)	2
  (0, 10)	48
  (0, 12)	1
  (0, 13)	10
  (0, 15)	1
  (0, 17)	1
  (0, 18)	3
  (0, 23)	1
  (0, 24)	6
  (0, 27)	1
  (0, 30)	5
  (0, 34)	3
  (0, 35)	12
  (0, 45)	25
  (0, 46)	2
  (0, 47)	1
  (0, 48)	1
  (0, 56)	1
  (0, 57)	2
  (0, 61)	1
  (0, 62)	2
  (0, 65)	1
  (0, 66)	1
  :	:
  (10, 21341)	3
  (10, 21441)	18
  (10, 21451)	1
  (10, 21474)	1
  (10, 21584)	1
  (10, 21585)	1
  (10, 21599)	1
  (10, 21603)	1
  (10, 21650)	2
  (10, 21671)	1
  (10, 21690)	2
  (10, 21731)	1
  (10, 21744)	4
  (10, 21893)	1
  (10, 21900)	1
  (10, 21929)	1
  (10, 22089)	3
  (10, 22100)	1
  (10, 22273)	4
  (10, 22327)	1
  (10, 22340)	1
  (10, 22439)	3
  (10, 22456)	1
  (10, 22463)	24
  (10, 22769)	1


### Decide train/test/validation split

In [3]:
splits = {
    "train": 0.7,
    "validation": 0.2,
    "test": 0.1
}

### Split within a user's data

This way the same user's data will be available to test on

In [4]:
rng = np.random.default_rng(1235)

sparse_matrix.sum_duplicates() # eliminate duplicates by adding them together
split_labels = np.zeros(sparse_matrix.nnz, dtype=int)

for u in range(sparse_matrix.shape[0]):
    start, end = sparse_matrix.indptr[u], sparse_matrix.indptr[u + 1]
    # number of artists
    num_artists = end - start
    # random order
    order = rng.permutation(num_artists)
    # assign indices
    num_artists_validation = int(splits["validation"] * num_artists)
    num_artists_test = int(splits["test"] * num_artists)
    # assign labels for validation and test (train is zero by default from np.zeros)
    for i in range(num_artists_validation):
        split_labels[start + order[i]] = 1
    for i in range(num_artists_test):
        split_labels[start + order[num_artists_validation + i]] = 2

def isolate_split(matrix, labels, label):
    split = matrix.copy()
    split.data[labels != label] = 0 # ignore cells from other splits
    split.eliminate_zeros() # remove those cells
    return split

train = isolate_split(sparse_matrix, split_labels, 0)
validation = isolate_split(sparse_matrix, split_labels, 1)
test = isolate_split(sparse_matrix, split_labels, 2)

In [11]:
train.shape

(11, 22823)

In [5]:
print(test)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5172 stored elements and shape (11, 22823)>
  Coords	Values
  (0, 34)	3
  (0, 48)	1
  (0, 81)	1
  (0, 104)	2
  (0, 107)	1
  (0, 120)	4
  (0, 123)	3
  (0, 139)	5
  (0, 183)	3
  (0, 184)	1
  (0, 206)	5
  (0, 220)	1
  (0, 229)	1
  (0, 243)	2
  (0, 282)	1
  (0, 314)	1
  (0, 322)	1
  (0, 340)	4
  (0, 363)	2
  (0, 367)	1
  (0, 376)	1
  (0, 388)	1
  (0, 392)	1
  (0, 398)	1
  (0, 430)	1
  :	:
  (10, 10815)	3
  (10, 10827)	1
  (10, 11620)	1
  (10, 12300)	1
  (10, 13335)	1
  (10, 14067)	1
  (10, 14524)	1
  (10, 15160)	1
  (10, 15439)	8
  (10, 15746)	2
  (10, 15777)	3
  (10, 15831)	1
  (10, 16269)	1
  (10, 16318)	1
  (10, 17296)	11
  (10, 17579)	1
  (10, 17999)	3
  (10, 18059)	1
  (10, 19047)	1
  (10, 19312)	1
  (10, 19876)	1
  (10, 20994)	1
  (10, 21327)	1
  (10, 21603)	1
  (10, 22089)	3
